!/usr/bin/env python
coding: utf-8


# 🧠 Mental Health in Tech Survey — Exploratory Data Analysis (EDA)
 
**Dataset:** 2014 survey measuring attitudes towards mental health and frequency of
mental health disorders in the tech workplace.
 
**1,261 respondents** from **41 countries**


In [ ]:

# ## 1. Setup & Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# ## 2. Load Data

df = pd.read_csv("survey.csv")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

# ## 3. Basic Info & Missing Values

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Missing Values ---")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False))



### Plot: Missing Values Heatmap


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(df.isnull().T, cbar=True, cmap='YlOrRd', yticklabels=True, ax=ax)
ax.set_title("Missing Values Heatmap", fontsize=16, fontweight='bold')
ax.set_xlabel("Row Index")
ax.set_ylabel("Columns")
plt.tight_layout()
plt.savefig("plots/01_missing_values_heatmap.png", bbox_inches='tight')
plt.show()


# ## 4. Data Cleaning

# ### 4a. Clean Age
print(f"\nAge before cleaning — Min: {df['Age'].min()}, Max: {df['Age'].max()}")
df['Age'] = df['Age'].apply(lambda x: x if 18 <= x <= 72 else np.nan)
df['Age'].dropna(inplace=False)
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Age'] = df['Age'].astype(int)
print(f"Age after cleaning — Min: {df['Age'].min()}, Max: {df['Age'].max()}")

# ### 4b. Clean Gender — Standardize to Male / Female / Other
def clean_gender(gender):
    if pd.isna(gender):
        return 'Other'
    gender = gender.strip().lower()
    male_terms = ['male', 'm', 'man', 'cis male', 'male ', 'maile', 'mal',
                  'male (cis)', 'make', 'guy (-ish) ^_^', 'msle', 'mail',
                  'cis man', 'male-ish', 'something kinda male?',
                  'male leaning androgynous', 'ostensibly male', 'malr']
    female_terms = ['female', 'f', 'woman', 'cis female', 'female ', 'femake',
                    'female (cis)', 'cis-female/femme', 'femail',
                    'female (trans)', 'trans-female', 'trans woman',
                    'queer/she/they']
    if gender in male_terms:
        return 'Male'
    elif gender in female_terms:
        return 'Female'
    else:
        return 'Other'

df['Gender'] = df['Gender'].apply(clean_gender)
print(f"\nGender distribution after cleaning:\n{df['Gender'].value_counts()}")

# ### 4c. Drop Timestamp & comments (not needed for analysis)
df.drop(['Timestamp', 'comments'], axis=1, inplace=True, errors='ignore')

# ### 4d. Fill NAs in self_employed
df['self_employed'] = df['self_employed'].fillna('No')

# ### 4e. Fill NAs in work_interfere
df['work_interfere'] = df['work_interfere'].fillna('Not applicable')

print(f"\nCleaned data shape: {df.shape}")
df.info()


# ## 5. Univariate Analysis

# ### 5a. Age Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Age'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Age'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['Age'].mean():.1f}")
axes[0].axvline(df['Age'].median(), color='orange', linestyle='--', linewidth=2, label=f"Median: {df['Age'].median():.0f}")
axes[0].set_title("Age Distribution", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
axes[0].legend()

sns.boxplot(x=df['Age'], ax=axes[1], color='steelblue')
axes[1].set_title("Age Boxplot", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Age")

plt.tight_layout()
plt.savefig("plots/02_age_distribution.png", bbox_inches='tight')
plt.show()

# ### 5b. Gender Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

gender_counts = df['Gender'].value_counts()
colors_gender = ['#3498db', '#e74c3c', '#2ecc71']
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=colors_gender, startangle=140, textprops={'fontsize': 12})
axes[0].set_title("Gender Distribution (Pie)", fontsize=14, fontweight='bold')

sns.countplot(x='Gender', data=df, order=gender_counts.index, palette=colors_gender, ax=axes[1])
axes[1].set_title("Gender Distribution (Bar)", fontsize=14, fontweight='bold')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.savefig("plots/03_gender_distribution.png", bbox_inches='tight')
plt.show()

# ### 5c. Country Distribution — Top 10
fig, ax = plt.subplots(figsize=(12, 6))
top_countries = df['Country'].value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, palette='viridis', ax=ax)
ax.set_title("Top 10 Countries by Respondent Count", fontsize=14, fontweight='bold')
ax.set_xlabel("Count")
ax.set_ylabel("Country")
for i, v in enumerate(top_countries.values):
    ax.text(v + 3, i, str(v), va='center', fontsize=11)
plt.tight_layout()
plt.savefig("plots/04_top_countries.png", bbox_inches='tight')
plt.show()

# ### 5d. Company Size Distribution
fig, ax = plt.subplots(figsize=(10, 5))
size_order = ['1-5', '6-25', '26-100', '100-500', '500-1000', 'More than 1000']
sns.countplot(x='no_employees', data=df, order=size_order, palette='coolwarm', ax=ax)
ax.set_title("Company Size Distribution", fontsize=14, fontweight='bold')
ax.set_xlabel("Number of Employees")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig("plots/05_company_size.png", bbox_inches='tight')
plt.show()

# ### 5e. Treatment Sought
fig, ax = plt.subplots(figsize=(7, 5))
treatment_counts = df['treatment'].value_counts()
ax.pie(treatment_counts, labels=treatment_counts.index, autopct='%1.1f%%',
       colors=['#27ae60', '#e74c3c'], startangle=90, textprops={'fontsize': 13})
ax.set_title("Have You Sought Treatment?", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("plots/06_treatment_pie.png", bbox_inches='tight')
plt.show()

# ### 5f. Work Interference
fig, ax = plt.subplots(figsize=(10, 5))
wi_order = ['Never', 'Rarely', 'Sometimes', 'Often', 'Not applicable']
sns.countplot(x='work_interfere', data=df, order=wi_order, palette='RdYlGn_r', ax=ax)
ax.set_title("Does Mental Health Condition Interfere with Work?", fontsize=14, fontweight='bold')
ax.set_xlabel("Level of Interference")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig("plots/07_work_interfere.png", bbox_inches='tight')
plt.show()

# ### 5g. Ease of Taking Leave
fig, ax = plt.subplots(figsize=(10, 5))
leave_order = ['Very easy', 'Somewhat easy', "Don't know", 'Somewhat difficult', 'Very difficult']
sns.countplot(x='leave', data=df, order=leave_order, palette='RdYlBu', ax=ax)
ax.set_title("How Easy is it to Take Mental Health Leave?", fontsize=14, fontweight='bold')
ax.set_xlabel("Ease of Leave")
ax.set_ylabel("Count")
plt.xticks(rotation=15)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig("plots/08_leave_ease.png", bbox_inches='tight')
plt.show()


# ## 6. Bivariate & Multivariate Analysis

# ### 6a. Treatment by Gender
fig, ax = plt.subplots(figsize=(8, 5))
ct = pd.crosstab(df['Gender'], df['treatment'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, color=['#e74c3c', '#27ae60'], ax=ax, edgecolor='white')
ax.set_title("Treatment Rate by Gender", fontsize=14, fontweight='bold')
ax.set_xlabel("Gender")
ax.set_ylabel("Percentage (%)")
ax.legend(title='Treatment', labels=['No', 'Yes'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots/09_treatment_by_gender.png", bbox_inches='tight')
plt.show()

# ### 6b. Age Distribution by Treatment
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(x='treatment', y='Age', data=df, palette=['#e74c3c', '#27ae60'],
               inner='quartile', ax=ax)
ax.set_title("Age Distribution by Treatment Status", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("plots/10_age_by_treatment_violin.png", bbox_inches='tight')
plt.show()

# ### 6c. Family History vs Treatment (Heatmap)
fig, ax = plt.subplots(figsize=(6, 4))
ct_fam = pd.crosstab(df['family_history'], df['treatment'])
sns.heatmap(ct_fam, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title("Family History vs Treatment (Counts)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("plots/11_family_vs_treatment_heatmap.png", bbox_inches='tight')
plt.show()

# ### 6d. Remote Work vs Treatment
fig, ax = plt.subplots(figsize=(8, 5))
ct_remote = pd.crosstab(df['remote_work'], df['treatment'], normalize='index') * 100
ct_remote.plot(kind='bar', color=['#e67e22', '#3498db'], ax=ax, edgecolor='white')
ax.set_title("Treatment Rate by Remote Work Status", fontsize=14, fontweight='bold')
ax.set_ylabel("Percentage (%)")
ax.legend(title='Treatment', labels=['No', 'Yes'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots/12_remote_vs_treatment.png", bbox_inches='tight')
plt.show()

# ### 6e. Benefits vs Treatment
fig, ax = plt.subplots(figsize=(8, 5))
ct_benefits = pd.crosstab(df['benefits'], df['treatment'], normalize='index') * 100
ct_benefits.plot(kind='bar', stacked=True, cmap='Set2', ax=ax, edgecolor='white')
ax.set_title("Treatment Rate by Benefits Availability", fontsize=14, fontweight='bold')
ax.set_ylabel("Percentage (%)")
ax.legend(title='Treatment')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots/13_benefits_vs_treatment.png", bbox_inches='tight')
plt.show()

# ### 6f. Work Interference by Gender
fig, ax = plt.subplots(figsize=(12, 5))
ct_wi = pd.crosstab(df['Gender'], df['work_interfere'], normalize='index') * 100
ct_wi = ct_wi[['Never', 'Rarely', 'Sometimes', 'Often', 'Not applicable']]
ct_wi.plot(kind='bar', stacked=True, cmap='RdYlGn_r', ax=ax, edgecolor='white')
ax.set_title("Work Interference Levels by Gender", fontsize=14, fontweight='bold')
ax.set_ylabel("Percentage (%)")
ax.legend(title='Interference', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots/14_work_interfere_by_gender.png", bbox_inches='tight')
plt.show()

# ### 6g. Mental Health Consequence Fear by Company Size
fig, ax = plt.subplots(figsize=(12, 6))
ct_cons = pd.crosstab(df['no_employees'], df['mental_health_consequence'], normalize='index') * 100
ct_cons = ct_cons.reindex(size_order)
ct_cons.plot(kind='bar', stacked=True, color=['#2ecc71', '#f39c12', '#e74c3c'], ax=ax, edgecolor='white')
ax.set_title("Fear of Negative Consequences by Company Size", fontsize=14, fontweight='bold')
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("Company Size")
ax.legend(title='Consequence')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig("plots/15_consequence_by_company_size.png", bbox_inches='tight')
plt.show()

# ### 6h. Supervisor Willingness by Benefits
fig, ax = plt.subplots(figsize=(10, 5))
ct_sup = pd.crosstab(df['benefits'], df['supervisor'], normalize='index') * 100
ct_sup.plot(kind='bar', cmap='Set1', ax=ax, edgecolor='white')
ax.set_title("Willingness to Discuss with Supervisor by Benefits", fontsize=14, fontweight='bold')
ax.set_ylabel("Percentage (%)")
ax.legend(title='Supervisor Discussion')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots/16_supervisor_by_benefits.png", bbox_inches='tight')
plt.show()


# ## 7. Correlation & Association Analysis

# ### 7a. Encode categorical variables for correlation
binary_map = {'Yes': 1, 'No': 0}
df_encoded = df.copy()

binary_cols = ['self_employed', 'family_history', 'treatment', 'remote_work',
               'tech_company', 'obs_consequence']
for col in binary_cols:
    df_encoded[col] = df_encoded[col].map(binary_map)

# Encode ordinal / multi-class columns
wi_map = {'Not applicable': 0, 'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4}
df_encoded['work_interfere'] = df_encoded['work_interfere'].map(wi_map)

leave_map = {"Don't know": 0, 'Very difficult': 1, 'Somewhat difficult': 2,
             'Somewhat easy': 3, 'Very easy': 4}
df_encoded['leave'] = df_encoded['leave'].map(leave_map)

tristate_map = {'No': 0, 'Maybe': 1, 'Yes': 2}
tri_cols = ['mental_health_consequence', 'phys_health_consequence', 'coworkers',
            'supervisor', 'mental_health_interview', 'phys_health_interview']
for col in tri_cols:
    df_encoded[col] = df_encoded[col].map(tristate_map)

tristate2_map = {'No': 0, "Don't know": 1, 'Yes': 2}
tri2_cols = ['benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity']
for col in tri2_cols:
    df_encoded[col] = df_encoded[col].map(tristate2_map)

mvp_map = {"Don't know": 0, 'No': 1, 'Yes': 2}
df_encoded['mental_vs_physical'] = df_encoded['mental_vs_physical'].map(mvp_map)

gender_map = {'Male': 0, 'Female': 1, 'Other': 2}
df_encoded['Gender'] = df_encoded['Gender'].map(gender_map)

# Select numeric columns only
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
cols_to_remove = ['Timestamp'] if 'Timestamp' in numeric_cols else []
numeric_cols = [c for c in numeric_cols if c not in cols_to_remove]

# ### 7b. Full Correlation Heatmap
fig, ax = plt.subplots(figsize=(18, 14))
corr_matrix = df_encoded[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            annot_kws={'size': 8})
ax.set_title("Correlation Heatmap of All Features", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig("plots/17_correlation_heatmap.png", bbox_inches='tight')
plt.show()

# ### 7c. Top Correlations with Treatment
treatment_corr = corr_matrix['treatment'].drop('treatment').sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#27ae60' if v > 0 else '#e74c3c' for v in treatment_corr.values]
treatment_corr.plot(kind='barh', color=colors, ax=ax, edgecolor='white')
ax.set_title("Feature Correlations with Treatment", fontsize=14, fontweight='bold')
ax.set_xlabel("Correlation Coefficient")
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig("plots/18_treatment_correlations.png", bbox_inches='tight')
plt.show()


# ## 8. Mental Health Attitudes Deep Dive

# ### 8a. Would you discuss MH with coworkers vs supervisor?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='coworkers', data=df, order=['No', 'Some of them', 'Yes'],
              palette='Set2', ax=axes[0])
axes[0].set_title("Discuss MH with Coworkers?", fontsize=13, fontweight='bold')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

sns.countplot(x='supervisor', data=df, order=['No', 'Some of them', 'Yes'],
              palette='Set2', ax=axes[1])
axes[1].set_title("Discuss MH with Supervisor?", fontsize=13, fontweight='bold')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig("plots/19_discuss_coworkers_supervisor.png", bbox_inches='tight')
plt.show()

# ### 8b. Interview Disclosure: Mental vs Physical
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mhi = df['mental_health_interview'].value_counts()
axes[0].pie(mhi, labels=mhi.index, autopct='%1.1f%%', colors=['#e74c3c', '#f39c12', '#27ae60'],
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title("Bring Up Mental Health in Interview?", fontsize=13, fontweight='bold')

phi = df['phys_health_interview'].value_counts()
axes[1].pie(phi, labels=phi.index, autopct='%1.1f%%', colors=['#e74c3c', '#f39c12', '#27ae60'],
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title("Bring Up Physical Health in Interview?", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig("plots/20_interview_mh_vs_ph.png", bbox_inches='tight')
plt.show()

# ### 8c. Observed Negative Consequences
fig, ax = plt.subplots(figsize=(7, 5))
obs = df['obs_consequence'].value_counts()
sns.barplot(x=obs.index, y=obs.values, palette=['#27ae60', '#e74c3c'], ax=ax)
ax.set_title("Observed Negative Consequences for MH?", fontsize=14, fontweight='bold')
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.savefig("plots/21_observed_consequences.png", bbox_inches='tight')
plt.show()

# ### 8d. Mental vs Physical: Employer Seriousness
fig, ax = plt.subplots(figsize=(8, 5))
mvp = df['mental_vs_physical'].value_counts()
sns.barplot(x=mvp.index, y=mvp.values, palette='rocket', ax=ax)
ax.set_title("Does Employer Take MH as Seriously as Physical Health?", fontsize=13, fontweight='bold')
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.savefig("plots/22_mental_vs_physical.png", bbox_inches='tight')
plt.show()


# ## 9. Geographic Analysis

# ### 9a. Treatment Rate by Top 10 Countries
fig, ax = plt.subplots(figsize=(12, 6))
top10 = df['Country'].value_counts().head(10).index
df_top10 = df[df['Country'].isin(top10)]
ct_country = pd.crosstab(df_top10['Country'], df_top10['treatment'], normalize='index') * 100
ct_country = ct_country.sort_values('Yes', ascending=True)
ct_country.plot(kind='barh', stacked=True, color=['#e74c3c', '#27ae60'], ax=ax, edgecolor='white')
ax.set_title("Treatment Rate by Country (Top 10)", fontsize=14, fontweight='bold')
ax.set_xlabel("Percentage (%)")
ax.legend(title='Treatment')
plt.tight_layout()
plt.savefig("plots/23_treatment_by_country.png", bbox_inches='tight')
plt.show()

# ### 9b. US State Analysis — Top 15 States
fig, ax = plt.subplots(figsize=(12, 6))
us_df = df[df['Country'] == 'United States']
top_states = us_df['state'].value_counts().head(15)
sns.barplot(x=top_states.values, y=top_states.index, palette='magma', ax=ax)
ax.set_title("Top 15 US States by Respondent Count", fontsize=14, fontweight='bold')
ax.set_xlabel("Count")
for i, v in enumerate(top_states.values):
    ax.text(v + 0.5, i, str(v), va='center', fontsize=11)
plt.tight_layout()
plt.savefig("plots/24_us_states.png", bbox_inches='tight')
plt.show()


# ## 10. Workplace Support Analysis

# ### 10a. Employer Resources Overview
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
support_cols = ['benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'leave']
titles = ['MH Benefits Provided?', 'Know Care Options?', 'Wellness Program?',
          'Employer Provides MH Resources?', 'Anonymity Protected?', 'Ease of Leave']

for i, (col, title) in enumerate(zip(support_cols, titles)):
    row, col_idx = divmod(i, 3)
    ax = axes[row][col_idx]
    vc = df[col].value_counts()
    vc.plot(kind='bar', color=sns.color_palette('pastel', len(vc)), ax=ax, edgecolor='gray')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel("Count")
    plt.sca(ax)
    plt.xticks(rotation=30, ha='right')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=9)

plt.suptitle("Workplace Mental Health Support Overview", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("plots/25_workplace_support_overview.png", bbox_inches='tight')
plt.show()


# ## 11. Age Group Analysis

df['Age_Group'] = pd.cut(df['Age'], bins=[17, 25, 35, 45, 55, 75],
                          labels=['18-25', '26-35', '36-45', '46-55', '56+'])

# ### 11a. Treatment Rate by Age Group
fig, ax = plt.subplots(figsize=(10, 5))
ct_age = pd.crosstab(df['Age_Group'], df['treatment'], normalize='index') * 100
ct_age.plot(kind='bar', stacked=True, color=['#e74c3c', '#27ae60'], ax=ax, edgecolor='white')
ax.set_title("Treatment Rate by Age Group", fontsize=14, fontweight='bold')
ax.set_ylabel("Percentage (%)")
ax.legend(title='Treatment')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots/26_treatment_by_age_group.png", bbox_inches='tight')
plt.show()

# ### 11b. Work Interference by Age Group
fig, ax = plt.subplots(figsize=(12, 5))
ct_wi_age = pd.crosstab(df['Age_Group'], df['work_interfere'], normalize='index') * 100
wi_cols = ['Never', 'Rarely', 'Sometimes', 'Often', 'Not applicable']
ct_wi_age = ct_wi_age[[c for c in wi_cols if c in ct_wi_age.columns]]
ct_wi_age.plot(kind='bar', stacked=True, cmap='RdYlGn_r', ax=ax, edgecolor='white')
ax.set_title("Work Interference by Age Group", fontsize=14, fontweight='bold')
ax.set_ylabel("Percentage (%)")
ax.legend(title='Interference', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("plots/27_work_interfere_by_age.png", bbox_inches='tight')
plt.show()


# ## 12. Tech vs Non-Tech Company

# ### 12a. Treatment & Benefits in Tech vs Non-Tech
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct_tech_treat = pd.crosstab(df['tech_company'], df['treatment'], normalize='index') * 100
ct_tech_treat.plot(kind='bar', color=['#e74c3c', '#27ae60'], ax=axes[0], edgecolor='white')
axes[0].set_title("Treatment Rate: Tech vs Non-Tech", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Percentage (%)")
axes[0].legend(title='Treatment')
axes[0].set_xticklabels(['Non-Tech', 'Tech'], rotation=0)

ct_tech_ben = pd.crosstab(df['tech_company'], df['benefits'], normalize='index') * 100
ct_tech_ben.plot(kind='bar', cmap='Set2', ax=axes[1], edgecolor='white')
axes[1].set_title("Benefits Availability: Tech vs Non-Tech", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Percentage (%)")
axes[1].legend(title='Benefits')
axes[1].set_xticklabels(['Non-Tech', 'Tech'], rotation=0)

plt.tight_layout()
plt.savefig("plots/28_tech_vs_nontech.png", bbox_inches='tight')
plt.show()


# ## 13. Self-Employment Analysis

fig, ax = plt.subplots(figsize=(8, 5))
ct_self = pd.crosstab(df['self_employed'], df['treatment'], normalize='index') * 100
ct_self.plot(kind='bar', color=['#e74c3c', '#27ae60'], ax=ax, edgecolor='white')
ax.set_title("Treatment Rate: Self-Employed vs Employed", fontsize=14, fontweight='bold')
ax.set_ylabel("Percentage (%)")
ax.legend(title='Treatment')
ax.set_xticklabels(['Employed', 'Self-Employed'], rotation=0)
plt.tight_layout()
plt.savefig("plots/29_self_employed_treatment.png", bbox_inches='tight')
plt.show()


# ## 14. Summary Dashboard

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top-left: Treatment pie
axes[0, 0].pie(df['treatment'].value_counts(), labels=['Yes', 'No'], autopct='%1.1f%%',
               colors=['#27ae60', '#e74c3c'], startangle=90, textprops={'fontsize': 14})
axes[0, 0].set_title("Sought Treatment", fontsize=14, fontweight='bold')

# Top-right: Family history vs treatment
ct_fam2 = pd.crosstab(df['family_history'], df['treatment'], normalize='index') * 100
ct_fam2.plot(kind='bar', stacked=True, color=['#e74c3c', '#27ae60'], ax=axes[0, 1], edgecolor='white')
axes[0, 1].set_title("Treatment by Family History", fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel("Percentage (%)")
axes[0, 1].legend(title='Treatment')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

# Bottom-left: Age distribution
axes[1, 0].hist(df['Age'], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
axes[1, 0].set_title("Age Distribution", fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel("Age")
axes[1, 0].set_ylabel("Count")

# Bottom-right: Top 5 countries
top5 = df['Country'].value_counts().head(5)
sns.barplot(x=top5.values, y=top5.index, palette='viridis', ax=axes[1, 1])
axes[1, 1].set_title("Top 5 Countries", fontsize=14, fontweight='bold')
for i, v in enumerate(top5.values):
    axes[1, 1].text(v + 3, i, str(v), va='center', fontsize=12)

plt.suptitle("Mental Health in Tech — Key Insights Dashboard", fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("plots/30_summary_dashboard.png", bbox_inches='tight')
plt.show()


# ## 15. Key Findings

print("""
╔══════════════════════════════════════════════════════════════╗
║                   KEY FINDINGS SUMMARY                       ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. ~50% of respondents have sought mental health treatment  ║
║  2. ~80% of respondents are Male                             ║
║  3. Family history is a strong predictor of seeking treatment ║
║  4. Most respondents work in companies with 26-100 employees ║
║  5. "Sometimes" is the most common work interference level   ║
║  6. Most people DON'T KNOW how easy it is to take MH leave  ║
║  7. Only ~5% would bring up MH in a job interview           ║
║  8. Females have a higher treatment-seeking rate than males  ║
║  9. US dominates the survey (~60%), followed by UK (~15%)    ║
║ 10. Tech companies offer slightly more MH benefits           ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝


)

print("✅ EDA Complete! All plots saved in the 'plots/' directory.")
